<a href="https://colab.research.google.com/github/roshjaison03/SmolLM-360M-Instruct-finetuning-with-Medical-Reasoning-data/blob/main/SmolLM_360M_Instruct_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shutil
import os

# Define the directory to remove
# directory_to_remove = "/content/peft-dialogue-response -training"

# Check if the directory exists and remove it
if os.path.exists(directory_to_remove) and os.path.isdir(directory_to_remove):
    shutil.rmtree(directory_to_remove)
    print(f"Directory '{directory_to_remove}' removed successfully.")
else:
    print(f"Directory '{directory_to_remove}' does not exist or is not a directory.")


directory_to_remove = "./peft-response"

if os.path.exists(directory_to_remove) and os.path.isdir(directory_to_remove):
    shutil.rmtree(directory_to_remove)
    print(f"Directory '{directory_to_remove}' removed successfully.")
else:
    print(f"Directory '{directory_to_remove}' does not exist or is not a directory.")

In [ ]:
%%capture
!pip install datasets
!pip install transformers
!pip install evaluate
!pip install accelerate - U
!pip install transformers[torch]
!pip install peft

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer, GenerationConfig
import evaluate
import pandas as pd
import numpy as np

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en")

In [ ]:
dataset['train'][3]

In [ ]:
del tokenizer

In [ ]:
device = "cuda"

In [ ]:
# Load model directly
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "HuggingFaceTB/SmolLM-360M-Instruct"
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-360M-Instruct", revision="v0.1").to(device)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
i = 20
dialogue = dataset['train'][i]['Question']
cot = dataset['train'][i]['Complex_CoT']
summary = dataset['train'][i]['Response']


In [ ]:
# Proper prompt construction for inference
messages = [
    {"role": "user", "content": dialogue}
]

# Use the tokenizer's chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    padding=True,
)

input_ids = inputs.input_ids.to(device)
attention_mask = inputs.attention_mask.to(device)

# Define generation_config
generation_config = GenerationConfig(max_new_tokens=200, temperature=0.7, do_sample=True, top_p=0.9, repetition_penalty=1.2)

outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    generation_config=generation_config,
)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
prompt

In [ ]:


input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
output = tokenizer.decode(model.generate(input_ids, max_new_tokens=200)[0], skip_special_tokens=True)

print(f"Input Prompt : {prompt}")
print("--------------------------------------------------------------------")
print("Human evaluated summary ---->")
print(summary)
print("---------------------------------------------------------------------")
print("Baseline model generated summary : ---->")
print(output)

In [ ]:
def tokenize_function(examples):
    """Train model to generate CoT reasoning then final answer."""
    MAX_LENGTH = 1024  # Use longer context for reasoning + answer

    inputs = []
    attention_masks = []
    all_labels = []

    for question, cot, response in zip(examples["Question"], examples["Complex_CoT"], examples["Response"]):
        # Create conversation: System -> User -> Assistant (CoT) -> Assistant (Answer)
        messages = [
            {
                "role": "system",
                "content": "You are an expert assistant. Think step by step and provide a clear reasoning before your final answer."
            },
            {"role": "user", "content": question},
            {"role": "assistant", "content": cot},      # TEACH: How to think
            {"role": "assistant", "content": response}  # TEACH: Final answer
        ]

        # Apply chat template
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            truncation=False,
            add_generation_prompt=False
        )

        # Pad/truncate
        if len(formatted) > MAX_LENGTH:
            formatted = formatted[:MAX_LENGTH]
        else:
            formatted = formatted + [tokenizer.pad_token_id] * (MAX_LENGTH - len(formatted))

        # Create attention mask
        attention_mask = [1 if token != tokenizer.pad_token_id else 0 for token in formatted]

        # Create labels: -100 for context, actual tokens for assistant turns
        labels = [-100] * MAX_LENGTH

        # Find where first assistant message (CoT) starts
        # Tokenize just system + user to find the boundary
        context_messages = messages[:2]  # system + user
        context_encoded = tokenizer.apply_chat_template(
            context_messages,
            tokenize=True,
            add_generation_prompt=True  # Adds assistant token
        )
        assistant_start_idx = len(context_encoded)

        # Tokenize CoT + response together
        assistant_content = f"{cot}\n\nFinal Answer: {response}"
        assistant_encoded = tokenizer.encode(
            assistant_content,
            add_special_tokens=False
        )

        # Fill labels for assistant content
        for i, token in enumerate(assistant_encoded):
            if assistant_start_idx + i < MAX_LENGTH:
                labels[assistant_start_idx + i] = token

        inputs.append(formatted)
        attention_masks.append(attention_mask)
        all_labels.append(labels)

    return {
        "input_ids": inputs,
        "attention_mask": attention_masks,
        "labels": all_labels
    }
# Set the pad token for the tokenizer
# Set pad token
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the dataset
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['Question','Complex_CoT', 'Response']  # Remove original columns
)
# Removed the erroneous line: tokenized_datasets = dataset.filter()

# Check what we have
print(f"Dataset structure: {tokenized_datasets}")
print(f"First example keys: {tokenized_datasets['train'][0].keys()}")
print(f"Input IDs length: {len(tokenized_datasets['train'][0]['input_ids'])}")
print(f"Labels length: {len(tokenized_datasets['train'][0]['labels'])}")

In [ ]:
tokenized_datasets['train'][0]

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}\nall model parameters: {all_model_params}\npercentage of trainable model parameters: {100 * trainable_model_params / all_model_params:.2f}%"

print(print_number_of_trainable_model_parameters(model))

In [ ]:
output_dir = "/content/drive/MyDrive/smol_medical_response"

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=1,
)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Use CAUSAL_LM for autoregressive models like SmolLM/GPT2
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,  # Changed from SEQ_2_SEQ_LM
    r=8,  # Lower r for better stability
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]  # Specify target modules
)

peft_model_train = get_peft_model(model, lora_config)
print(print_number_of_trainable_model_parameters(peft_model_train))

In [ ]:
from transformers import DataCollatorForLanguageModeling

# Set pad token
tokenizer.pad_token = tokenizer.eos_token

# Create data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
    pad_to_multiple_of=8  # Optional: for GPU efficiency
)

peft_trainer = Trainer(
    model=peft_model_train,
    args=peft_training_args,
    train_dataset=tokenized_datasets['train'],
    data_collator=data_collator,  # Add this
)
peft_trainer.train()

In [ ]:
peft_model_path = "./peft-response"
peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

In [ ]:
device='cpu'
peft_model_train.to(device)

In [ ]:
# User-based prompt code to test the model

# Get user input for dialogue
dialogue = "Hello Doctor,I’ve been experiencing a persistent headache along with stomach discomfort since today. I also feel a bit fatigued and uneasy overall. I’m not sure what’s causing this and wanted to ask for your guidance on what I should do or if any medication or tests are needed. Thank you for your help."
import time
start = time.time()

# Create proper prompt
messages = [{"role": "user", "content": dialogue}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenize
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Generate with PEFT model
peft_model_train.eval()
with torch.no_grad():
    outputs = peft_model_train.generate(
        input_ids=input_ids,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.2
    )

peft_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract just the assistant's response
if "assistant" in peft_response:
    peft_response = peft_response.split("assistant")[-1].strip()
print(f"Time taken: {time.time() - start}")
print("="*50)
print("INPUT:", dialogue)
print("="*50)
print("PEFT MODEL RESPONSE:", peft_response)
print("="*50)